# 3 — Fresh cue-only CEBRA-Time and matched references
Independent source reconstruction; no old encoder, standardized array or notebook cache. Existing offset1-model/temperature1/Adam and exact row-uniform native-adjacent-bin Time sampler. Public demo uses20updates; reviewed biological configuration requires1000, unchanged.

In [ ]:
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')
from neuro.notebook_inputs import configuration_from_environment, inspect_inputs, load_inputs, effective_pair_map
from neuro.pseudopopulation import sample_plan, materialize, support
from neuro.notebook_artifacts import ArtifactStore, FitAccounting, target_heads, write_private_json, verify_artifacts, runtime_identity, validate_kernel_identity
from neuro.notebook_support import workspace, usage_summary, source_budget, checked_assemblies, compact_metrics, plot_confusions

configuration = configuration_from_environment()
settings = configuration['execution']
output = workspace()
def private_warning(message, category, filename, lineno, file=None, line=None):
    # Preserve full provenance privately; do not render host paths in shared figures/output.
    warning_path = output / 'warnings-private.jsonl'
    with warning_path.open('a') as handle:
        handle.write(json.dumps({'message': str(message), 'category': category.__name__, 'filename': filename, 'lineno': lineno}) + '\n')
    warning_path.chmod(0o600)
    print(category.__name__ + ': ' + str(message), file=sys.stderr)
warnings.showwarning = private_warning
store = ArtifactStore(output / 'artifacts')
store.save('configuration', configuration)
store.save('kernel_runtime_before_inputs', validate_kernel_identity(configuration))
print('Input mode:', configuration['mode'], '| exploratory computational assay; no biological IID inference')

## Explicit metadata and target boundary
Authenticate only supplied files, regenerate eligibility/assignments, freeze the contributor embargo and complete-support panel before neural interpretation. The public default is invented data. Missing supplied inputs fail; no demo or historical-result fallback.

In [ ]:
inspection = inspect_inputs(configuration)  # Metadata/hash step only for supplied inputs.
store.save('metadata_selection', inspection)  # Protected provenance; never displayed.
print(json.dumps(inspection.audit, sort_keys=True))
assert inspection.audit['contributor_overlap'] == 0
assert inspection.audit['selected_train'] > 0 and inspection.audit['selected_development'] > 0

In [ ]:
data, neural_audit = load_inputs(inspection)  # Supplied neural path remains explicitly review-gated.
store.save('source_data', data)
store.save('neural_audit', neural_audit)
print(json.dumps(neural_audit, sort_keys=True))
assert np.isfinite(data.values[data.observed]).all()
assert np.isnan(data.values[~data.observed]).all()
assert not data.dependency_audit()['overlapping_contributor_ids']
print('Source × unit × bin shape:', data.values.shape, '| units:', data.value_unit, '| clock units:', data.time_unit)

## Declare optimization, temporal pairs and worlds
The permutation shuffles joint pair labels at source level within block×split before matching. It preserves observed provenance and uses its own fixed development world. It is an alignment-breaking diagnostic, not a statistical null.

In [ ]:
import neuro.cebra as cebra
validate_kernel_identity(configuration)  # Includes actual optional dependency origins/code before fitting.
from neuro.pca_lda import fit_standardizer, raw_features, fit_pca
from neuro.notebook_support import fit_logistic
assert settings['seeds'] == [0, 1]
print({k: settings[k] for k in ('iterations', 'batch_size', 'hidden', 'output_dimension', 'learning_rate')})
print('Native positives:', [[b, b + 1] for b in range(data.values.shape[2] - 1)], '| references row-uniform; negatives row/bin-uniform')
accounting = FitAccounting(cebra=True).start()
results = []

## Visible fresh encoder, reference and readout fits
Every source-budget/world/seed refits normalization, PCA, CEBRA and three separate logistic heads. Development transformations cannot refit. Paired seeds do not represent biological replication.

In [ ]:
for world in ('observed', 'permuted'):
    world_map = effective_pair_map(data, permutation_seed=None if world == 'observed' else settings['permutation_seed'])
    development_plan = sample_plan(data, split_id='development', rows_per_condition=settings['development_rows_per_condition'], seed=0, effective_conditions=world_map)
    reference_development = materialize(data, development_plan)
    store.save(world + '_fixed_development', reference_development)
    for seed in (settings['seeds'] if world == 'observed' else [0]):
        for budget in ('full', 'reduced'):
            selected, effective = source_budget(data, world_map, reduced=budget == 'reduced', subset_seed=settings['subset_seed'])
            train, development = checked_assemblies(selected, effective, development_plan, reference_development, config=settings, seed=seed)
            key = f'{world}_seed{seed}_{budget}'
            store.save_bundle(key, train, development)
            print(key, 'source accounting:', json.dumps(usage_summary(selected, train), sort_keys=True))
            standardizer = fit_standardizer(train.values)
            raw_train, raw_dev = raw_features(standardizer.transform(train.values)), raw_features(standardizer.transform(development.values))
            pca = fit_pca(train.values, n_components=settings['pca_components'])
            remaining = float(os.environ.get('NEURO_DEADLINE_MONOTONIC', time.monotonic() + 120)) - time.monotonic()
            encoder = cebra.fit(train.values, train.targets, iterations=settings['iterations'], batch_size=settings['batch_size'],
                                output_dimension=settings['output_dimension'], hidden=settings['hidden'], learning_rate=settings['learning_rate'],
                                seed=seed, max_seconds=remaining, device='cpu')
            store.save_encoder(key + '_raw', standardizer, kind='raw')
            store.save_encoder(key + '_pca', pca, kind='pca')
            store.save_encoder(key + '_cebra', encoder, kind='cebra')
            feature_pairs = {'raw': (raw_train, raw_dev), 'pca': (pca.transform(train.values), pca.transform(development.values)),
                             'cebra': (encoder.transform(train.values), encoder.transform(development.values))}
            run_results = []
            for representation, (train_x, dev_x) in feature_pairs.items():
                for head, y in target_heads(train).items():
                    model = fit_logistic(train_x, y, C=settings['logistic_C'], max_iter=settings['logistic_max_iter'])
                    row = store.readout(key + '_' + representation + '_' + head, model, encoder=key + '_' + representation,
                                        bundle=key, head=head, train_features=train_x, development_features=dev_x)
                    run_results.append(row)
            results.extend(run_results)
            print(json.dumps(compact_metrics(run_results), sort_keys=True))
            print('Realized sampler phase exposure:', encoder.diagnostics['exposure_summary']['reference_phase_counts'])
            assert len(encoder.diagnostics['loss']['total']) == settings['iterations']
            assert np.isfinite(encoder.diagnostics['loss']['total']).all()
            fig, axes = plt.subplots(1, 2, figsize=(9, 3))
            axes[0].plot(encoder.diagnostics['loss']['total']); axes[0].set(xlabel='Optimizer update', ylabel='InfoNCE loss', title=key)
            axes[1].hist(encoder.diagnostics['sample_exposure_counts'].reshape(-1), bins=15)
            axes[1].set(xlabel='Realized reference/positive/negative uses', ylabel='Native vector entries', title='Sampler exposure, not effective biological n')
            display(fig); plt.close(fig)
            fig = plot_confusions(run_results, title=key + ': development counts and row-normalized confusions')
            display(fig); plt.close(fig)
store.save('aggregate_metrics', results)

## Inspect transformed held-out coordinates
This descriptive plot does not establish neural-state dynamics or override matched readout scores.

In [ ]:
z = feature_pairs['cebra'][1]
fig, ax = plt.subplots(figsize=(5, 3)); ax.scatter(z[:, 0], z[:, 1], s=6)
ax.set(xlabel='CEBRA coordinate1, first cue bin', ylabel='CEBRA coordinate2, first cue bin', title='Transformed development (descriptive)')
display(fig); plt.close(fig)
print('Grid summary: 6CEBRA,6PCA,54logistic; observed and permuted worlds are not pooled.')

## Interpretation limits
These are source-safe conditional pseudopopulations, not simultaneous trials. Matching uses known labels; this is not an unknown-trial prediction pipeline. Bins, synthetic rows and computational seeds are dependent. Place-selected units, provisional marker meaning, inherited preprocessing and historical exposure remain limitations. Contributor separation does not remove long-range serial/session confounds. No biological confidence intervals, p-values, mechanism, dynamics or prospective-path claim follows from these figures. The permutation diagnostic is alignment-breaking, not a biological exchangeability test.

In [ ]:
fit_counts = accounting.stop()
store.save('kernel_runtime_final', runtime_identity())
expected_counts = {'pca': 6, 'logistic': 54, 'lda': 0, 'cebra': 6, 'optimizer_updates': 6 * settings['iterations']}
manifest_hash = store.finish(fit_counts=fit_counts, expected_counts=expected_counts)
replay = verify_artifacts(output / 'artifacts', expected_manifest_hash=manifest_hash)
write_private_json(output / 'completion.json', {'manifest_sha256': manifest_hash, 'fit_counts': fit_counts, 'replay': replay})
print(json.dumps(replay, sort_keys=True))
assert replay['status'] == 'fit-blocked-loaded-model-replay-passed'